# 🔬 SciCLIP: LoRA-Adapted CLIP for Scientific Figure Retrieval

> **Runtime → Change runtime type → A100 GPU** for best performance (~2 hrs full training)  
> T4 also works but slower (~5 hrs)

This notebook walks through the full SciCLIP pipeline:
1. Install dependencies
2. Clone the repo & prepare data
3. Train LoRA-CLIP (or run a quick mini-test)
4. Evaluate: Vanilla CLIP vs LoRA-CLIP
5. Build FAISS index & run retrieval demo

## 0. Check GPU

In [ ]:
import torch
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected — go to Runtime → Change runtime type → GPU")

## 1. Install Dependencies

In [ ]:
!pip install -q \
    "transformers>=4.38.0,<4.46.0" \
    "peft>=0.9.0,<0.15.0" \
    "datasets>=2.18.0" \
    "gradio>=3.50.0" \
    "accelerate>=0.27.0" \
    "huggingface-hub>=0.23.2,<1.0" \
    wandb tqdm

import subprocess, sys, torch
def install_faiss():
    if not torch.cuda.is_available():
        print("No GPU → installing faiss-cpu")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"])
        return
    cuda_ver = torch.version.cuda        # e.g. "12.1" or "11.8"
    major = int(cuda_ver.split(".")[0]) if cuda_ver else 0
    if major >= 12:
        pkg = "faiss-gpu-cu12"
    elif major == 11:
        pkg = "faiss-gpu-cu11"
    else:
        pkg = "faiss-cpu"
    print(f"CUDA {cuda_ver} detected → installing {pkg}")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True
    )
    if result.returncode != 0:
        print(f"{pkg} failed → falling back to faiss-cpu")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"])
install_faiss()

print("✅ Dependencies installed")

## 2. Clone SciCLIP Repo

In [ ]:
import os

REPO_URL = "https://github.com/jkhyjkhy/sciclip.git"

if not os.path.exists("sciclip"):
    !git clone {REPO_URL}
    %cd sciclip
else:
    %cd sciclip
    !git pull origin main
!ls

## 3. Download & Prepare SciCap Data

- **Full run**: `--max_samples 20000` (~20 min download)
- **Mini test**: `--max_samples 200` (~1 min, just to verify pipeline)

In [ ]:
# ✏️ Set this to 200 for a quick sanity check, 20000 for full training
MAX_SAMPLES = 200  # change to 20000 for full run

In [ ]:
# Insert this cell after the MAX_SAMPLES cell
from google.colab import drive
import os
import zipfile
import shutil

# Mount Google Drive
try:
    drive.mount('/content/drive')
    drive_mounted = True
except Exception as e:
    print(f"Google Drive mount bypassed: {e}")
    drive_mounted = False

DATA_RESTORED = False

if drive_mounted:
    backup_file = f'/content/drive/MyDrive/SciCLIP/scicap_{MAX_SAMPLES}.zip'
    local_zip = f'/content/scicap_{MAX_SAMPLES}.zip'
    target_dir = '/content/sciclip/data'
    
    if os.path.exists(backup_file):
        print(f"✓ Found backup file for {MAX_SAMPLES} samples on Google Drive.")
        print("Copying backup archive to local scratch disk...")
        shutil.copy(backup_file, local_zip)
        
        print("Extracting preprocessed dataset...")
        os.makedirs(target_dir, exist_ok=True)
        with zipfile.ZipFile(local_zip, 'r') as zip_ref:
            zip_ref.extractall(target_dir)
            
        print("✓ Dataset successfully restored from Google Drive backup.")
        DATA_RESTORED = True
    else:
        print(f"No backup file found at '{backup_file}'. Proceeding to download fresh dataset.")

In [ ]:
# Insert this cell after the Restore cell
import os
from google.colab import userdata

if not DATA_RESTORED:
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        print("✓ HF_TOKEN loaded from Colab Secrets and injected into environment.")
    except Exception as e:
        print(f"⚠️ Could not load HF_TOKEN: {e}")
        print("Ensure 'HF_TOKEN' is set in Colab Secrets and notebook access is enabled.")
else:
    print("Dataset already restored from backup. HF_TOKEN setup skipped.")

In [ ]:
# Replace your existing dataset preparation cell with this
if not DATA_RESTORED:
    # Run the updated dataset preparation script
    !python data/prepare_scicap.py --max_samples {MAX_SAMPLES} --output_dir data/
else:
    print("Skipping download and extraction (dataset restored from backup).")

# Verify the final JSONL splits
!echo "--- train ---" && wc -l data/scicap_train.jsonl
!echo "--- val ---"   && wc -l data/scicap_val.jsonl

In [ ]:
# Insert this cell after the data preparation execution cell
if drive_mounted and not DATA_RESTORED:
    import shutil
    
    backup_file = f'/content/drive/MyDrive/SciCLIP/scicap_{MAX_SAMPLES}.zip'
    local_zip = f'/content/scicap_{MAX_SAMPLES}.zip'
    
    print(f"Creating backup archive for {MAX_SAMPLES} samples...")
    shutil.make_archive(f'/content/scicap_{MAX_SAMPLES}', 'zip', '/content/sciclip/data')
    
    backup_dir = '/content/drive/MyDrive/SciCLIP'
    os.makedirs(backup_dir, exist_ok=True)
    
    print("Uploading backup archive to Google Drive...")
    shutil.copy(local_zip, backup_file)
    print(f"✓ Backup uploaded successfully to: {backup_file}")
elif DATA_RESTORED:
    print("Dataset was restored from backup. No new backup needed.")
else:
    print("Google Drive not mounted. Skipping backup creation.")

## 4. Train LoRA-CLIP

Key flags:
| Flag | Default | Description |
|------|---------|-------------|
| `--lora_r` | 8 | LoRA rank (ablation: 4, 8, 16) |
| `--epochs` | 5 | Training epochs |
| `--batch_size` | 64 | Batch size |
| `--wandb` | off | Enable W&B logging |
| `--ablation` | off | Run r=4,8,16 sequentially |

In [ ]:
# ✏️ Training config — adjust as needed
LORA_R      = 8
EPOCHS      = 5       # set to 1 for mini test
BATCH_SIZE  = 64
USE_WANDB   = False   # set True if you have a W&B account

In [ ]:
wandb_flag = "--wandb" if USE_WANDB else ""

!python train.py \
    --lora_r {LORA_R} \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    {wandb_flag}

In [ ]:
# Verify checkpoint was saved
!ls checkpoints/lora_r{LORA_R}/best_adapter/
!cat checkpoints/lora_r{LORA_R}/history.json

## 5. Ablation Study (LoRA Rank r = 4, 8, 16)

Skip this cell if you only want the r=8 result.

In [ ]:
# ⚠️  This runs 3× training — takes ~6 hrs on A100 with 20k samples
# Uncomment to run:

# !python train.py --ablation --epochs {EPOCHS} --batch_size {BATCH_SIZE}

## 6. Evaluate: Vanilla CLIP vs LoRA-CLIP

In [ ]:
!python evaluate.py \
    --adapter_path checkpoints/lora_r{LORA_R}/best_adapter \
    --val_path data/scicap_val.jsonl

In [ ]:
# Or run in-notebook for inline results
import sys, torch
sys.path.insert(0, '.')

from evaluate import compare_baseline_vs_lora

results = compare_baseline_vs_lora(
    adapter_path=f"checkpoints/lora_r{LORA_R}/best_adapter",
    val_path="data/scicap_val.jsonl",
)

## 7. Build FAISS Retrieval Index

In [ ]:
!python build_index.py \
    --adapter_path checkpoints/lora_r{LORA_R}/best_adapter \
    --data_path data/scicap_train.jsonl \
    --index_path data/sciclip.index

!ls -lh data/sciclip.index data/metadata.pkl data/embeddings.npy

## 8. CLI Retrieval Test

In [ ]:
# Test a few queries
queries = [
    "attention mechanism architecture diagram",
    "loss curve showing training convergence",
    "bar chart comparing model performance",
    "transformer encoder decoder architecture",
]

for q in queries:
    print(f"\n🔍 Query: '{q}'")
    !python retrieve.py \
        --query "{q}" \
        --adapter_path checkpoints/lora_r{LORA_R}/best_adapter \
        --index_path data/sciclip.index \
        --top_k 3

## 9. Gradio Demo

Launches a web UI at the public Gradio link.  
Shows **side-by-side comparison**: Vanilla CLIP vs LoRA-CLIP results.

In [ ]:
%cd /content/sciclip

In [ ]:
!ls -lh checkpoints/lora_r8/best_adapter/

In [ ]:
!python app.py \
    --adapter_path checkpoints/lora_r{LORA_R}/best_adapter \
    --index_path data/sciclip.index \
    --share  # generates public URL

## 10. Save Adapter to HuggingFace Hub (optional)

Upload the trained LoRA adapter so others can use it directly.

In [ ]:
# ✏️ Fill in your HuggingFace username and repo name
HF_USERNAME  = "jkhyjkhy"
HF_REPO_NAME = "sciclip-lora-r8"

from huggingface_hub import HfApi

api = HfApi()
api.create_repo(f"{HF_USERNAME}/{HF_REPO_NAME}", exist_ok=True)

api.upload_folder(
    folder_path=f"checkpoints/lora_r{LORA_R}/best_adapter",
    repo_id=f"{HF_USERNAME}/{HF_REPO_NAME}",
    repo_type="model",
)

print(f"✅ Adapter uploaded: https://huggingface.co/{HF_USERNAME}/{HF_REPO_NAME}")

## 11. Results Summary

After training, copy your results into the table below for the term paper.

In [ ]:
import json
from pathlib import Path

print("=" * 60)
print(f"{'Model':<20} {'R@1':>8} {'R@5':>8} {'R@10':>8} {'MRR':>8}")
print("-" * 60)

# Fill in after running evaluate.py
# Example structure — replace with your actual results:
table = [
    {"model": "Vanilla CLIP",     "r1": "—", "r5": "—", "r10": "—", "mrr": "—"},
    {"model": "LoRA-CLIP (r=4)",  "r1": "—", "r5": "—", "r10": "—", "mrr": "—"},
    {"model": "LoRA-CLIP (r=8)",  "r1": "—", "r5": "—", "r10": "—", "mrr": "—"},
    {"model": "LoRA-CLIP (r=16)", "r1": "—", "r5": "—", "r10": "—", "mrr": "—"},
]

for row in table:
    print(f"{row['model']:<20} {row['r1']:>8} {row['r5']:>8} {row['r10']:>8} {row['mrr']:>8}")

print("=" * 60)
print("\n→ Copy this into README.md table when results are ready!")